In [144]:
import pandas as pd
import numpy as np
import time

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [20]:
df = pd.read_csv(r'D:\Downloads\scenario_manifest.csv')

In [21]:

# Assuming your dataframe is 'df' and the column with these classes is 'taxonomy'
normal_mask = df['scenario_id'].str.startswith('normal_')

# Sample exactly 500 from each 'normal' class (stratified sampling)
df_normal_sampled = (
    df[normal_mask]
    .groupby('scenario_id', group_keys=False)
    .apply(lambda x: x.sample(n=500, random_state=42))
)

# Keep the non-normal classes as they are
df_others = df[~normal_mask]

# Combine and shuffle the final dataset
df_balanced = pd.concat([df_normal_sampled, df_others]).sample(frac=1, random_state=42).reset_index(drop=True)

C:\Users\bassa\AppData\Local\Temp\ipykernel_22908\2136101372.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=500, random_state=42))


In [22]:


print("\n=== CATEGORICAL VALUES ===")
lis = df_balanced.select_dtypes(include=['object']).columns[1:]
for col in lis:
        print(f"\n{col}: {df_balanced[col].nunique()} unique values")
        print(df_balanced[col].value_counts().head(21))


=== CATEGORICAL VALUES ===

scenario_id: 21 unique values
scenario_id
raf_alg_mismatch                     500
model_gray_rsa_mimicry               500
anomaly_pqc_over_tls12               500
normal_mlkem768_mldsa65_cert         500
net_high_loss_pqc                    500
normal_mlkem768_mldsa87_cert         500
raf_downgrade_overt                  500
anomaly_kem_mismatch                 500
net_high_jitter_classic              500
normal_mlkem1024_mldsa87_cert        500
normal_mlkem768_rsa_cert             500
threat_data_exfil                    500
normal_X25519_ecdsa_p256_cert        500
net_high_latency_pqc                 500
normal_X25519_rsa_cert               500
normal_X25519_rsa3072_cert           500
normal_mlkem1024_rsa_cert            500
normal_mlkem1024_mldsa65_cert        500
normal_X25519_ecdsa_p384_cert        500
anomaly_hybrid_pki                   500
raf_robustness_malformed_keyshare     10
Name: count, dtype: int64

taxonomy_class: 5 unique values
taxonomy_

In [23]:
temp = pd.read_csv(r'D:\Downloads\df_with_idx.csv')

In [24]:
temp = temp.drop(columns=['Unnamed: 0'])

In [25]:
temp

,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID,idx,subtype
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,0,0,...,1,0,0,1,1,train,eval_test_network,net_high_jitter_classic_run1,38510,high_jitter_classic
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,0,0,...,1,0,0,1,1,val,eval_test_network,net_high_jitter_classic_run10,38519,high_jitter_classic
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,0,0,...,1,0,0,1,1,train,eval_test_network,net_high_jitter_classic_run100,38609,high_jitter_classic
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,0,0,...,1,0,0,1,1,test,eval_test_network,net_high_jitter_classic_run101,38610,high_jitter_classic
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,0,0,...,1,0,0,1,1,test,eval_test_network,net_high_jitter_classic_run102,38611,high_jitter_classic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40005,285,0.0000,0.0000,0.60,0.598192,64.0,129,2.0,0,1,...,0,1,0,0,1,train,raf_test_robustness,raf_robustness_malformed_keyshare_run5,37504,malformed_keyshare
40006,285,0.0000,0.0000,0.07,0.066996,64.0,129,2.0,0,1,...,0,1,0,0,1,val,raf_test_robustness,raf_robustness_malformed_keyshare_run6,37505,malformed_keyshare
40007,285,0.0000,0.0000,2.91,2.910137,64.0,129,2.0,0,1,...,0,1,0,0,1,train,raf_test_robustness,raf_robustness_malformed_keyshare_run7,37506,malformed_keyshare
40008,285,0.0000,0.0000,0.64,0.639915,64.0,129,2.0,0,1,...,0,1,0,0,1,train,raf_test_robustness,raf_robustness_malformed_keyshare_run8,37507,malformed_keyshare


In [26]:
df_balanced['ID'] = df_balanced['ID'].str.replace('.pcap', '', regex=False)

In [27]:
df_balanced

,ID,scenario_id,taxonomy_class,subtype,operational_class,anomaly_label,run_idx
0,raf_alg_mismatch_run439,raf_alg_mismatch,Threat,raf_alg_mismatch,Misconfig,Anomaly,439
1,threat_data_exfil_run49,threat_data_exfil,Threat,data_exfil,Compliant_PQC,Anomaly,49
2,normal_X25519_ecdsa_p384_cert_run791,normal_X25519_ecdsa_p384_cert,ML,baseline_normal,Classical,Normal,791
3,threat_data_exfil_run335,threat_data_exfil,Threat,data_exfil,Compliant_PQC,Anomaly,335
4,normal_mlkem1024_mldsa65_cert_run1412,normal_mlkem1024_mldsa65_cert,ML,baseline_normal,Compliant_mlkem1024,Normal,1412
...,...,...,...,...,...,...,...
10005,anomaly_pqc_over_tls12_run235,anomaly_pqc_over_tls12,Threat,pqc_over_tls12,Misconfig,Anomaly,235
10006,anomaly_kem_mismatch_run192,anomaly_kem_mismatch,Threat,kem_mismatch,Misconfig,Anomaly,192
10007,anomaly_kem_mismatch_run391,anomaly_kem_mismatch,Threat,kem_mismatch,Misconfig,Anomaly,391
10008,normal_X25519_ecdsa_p384_cert_run1843,normal_X25519_ecdsa_p384_cert,ML,baseline_normal,Classical,Normal,1843


In [28]:
# The tilde (~) means "NOT". So this keeps rows where the ID is NOT in df
unmatched_df = temp[~temp['ID'].isin(df_balanced['ID'])]

In [29]:
unmatched_df

,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID,idx,subtype
4000,7643,4.9375,5.8947,1.61,5.234003,32.0,285,5330.0,0,0,...,1,0,0,1,0,val,ml_baseline_normal,normal_X25519_ecdsa_p256_cert_run1,28000,baseline_normal
4001,7642,4.8750,5.9346,0.76,3.362894,32.0,285,5330.0,0,0,...,1,0,0,1,0,val,ml_baseline_normal,normal_X25519_ecdsa_p256_cert_run10,28009,baseline_normal
4002,7643,4.8750,5.9346,0.98,3.352880,32.0,285,5330.0,0,0,...,1,0,0,1,0,test,ml_baseline_normal,normal_X25519_ecdsa_p256_cert_run100,28099,baseline_normal
4004,7644,4.8750,5.9056,0.90,3.625154,32.0,285,5330.0,0,0,...,1,0,0,1,0,train,ml_baseline_normal,normal_X25519_ecdsa_p256_cert_run1001,29000,baseline_normal
4005,7643,4.9375,5.8367,0.81,4.005194,32.0,285,5330.0,0,0,...,1,0,0,1,0,val,ml_baseline_normal,normal_X25519_ecdsa_p256_cert_run1002,29001,baseline_normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38994,10412,4.8125,5.8077,2.45,4.662037,1184.0,1429,5322.0,0,0,...,1,0,0,1,0,train,ml_baseline_normal,normal_mlkem768_rsa_cert_run994,7993,baseline_normal
38995,10412,4.8750,5.8367,2.36,4.964113,1184.0,1429,5322.0,0,0,...,1,0,0,1,0,test,ml_baseline_normal,normal_mlkem768_rsa_cert_run995,7994,baseline_normal
38996,10412,5.0000,5.8477,2.46,4.528999,1184.0,1429,5322.0,0,0,...,1,0,0,1,0,train,ml_baseline_normal,normal_mlkem768_rsa_cert_run996,7995,baseline_normal
38997,10412,4.7500,5.8837,2.31,4.876137,1184.0,1429,5322.0,0,0,...,1,0,0,1,0,train,ml_baseline_normal,normal_mlkem768_rsa_cert_run997,7996,baseline_normal


In [29]:
# Option 1: Filter 'temp' to only keep rows with IDs that exist in 'df'
df = temp[temp['ID'].isin(df_balanced['ID'])]
splits = df['split'].str.lower().str.strip()
train_mask = (splits == 'train')
val_mask = (splits == 'val')
test_mask = (splits == 'test')

In [30]:
df = df.drop(columns=[ 'split', 'taxonomy', 'ID' ,'idx' ,'subtype'])

In [31]:
df

,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e1b_tls_version_TLS1.0,e1b_ciphersuite_2,e1b_ciphersuite_54,e1b_ciphersuite_60,e2b_tls_version_SSL3.0,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,0,0,...,1,0,0,1,0,1,0,0,1,1
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,0,0,...,1,0,0,1,0,1,0,0,1,1
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,0,0,...,1,0,0,1,0,1,0,0,1,1
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,0,0,...,1,0,0,1,0,1,0,0,1,1
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,0,0,...,1,0,0,1,0,1,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40005,285,0.0000,0.0000,0.60,0.598192,64.0,129,2.0,0,1,...,0,1,0,0,1,0,1,0,0,1
40006,285,0.0000,0.0000,0.07,0.066996,64.0,129,2.0,0,1,...,0,1,0,0,1,0,1,0,0,1
40007,285,0.0000,0.0000,2.91,2.910137,64.0,129,2.0,0,1,...,0,1,0,0,1,0,1,0,0,1
40008,285,0.0000,0.0000,0.64,0.639915,64.0,129,2.0,0,1,...,0,1,0,0,1,0,1,0,0,1


In [35]:
# df.to_csv(r'D:\Downloads\df_short.csv')
lis = df.select_dtypes(include=['bool'])
for col in lis:
    df[col] = le.fit_transform(df[col])

In [152]:

# 1. LOAD DATA
DATA_PATH = r'D:\Downloads\df_short.csv'
df = pd.read_csv(DATA_PATH)

# 2. SEPARATE TARGET AND DROP METADATA
# Safely drop label and any leftover text/ID columns if they still exist
cols_to_drop = ['label']
for col in ['taxonomy', 'ID','Unnamed: 0']: 
    if col in df.columns:
        cols_to_drop.append(col)
        
X = df.drop(columns=cols_to_drop)
y = df['label']

# XGBoost handles NaNs natively, but converting to float32 is good practice
X = X.astype('float32')
y = y.astype('int32') 

# 3. SPLIT THE DATA (70% Train, 15% Validation, 15% Test)
# Step A: Split off 70% for Training, leaving 30% for a "temp" set
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step B: Split the remaining 30% in half to get 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 4. CALCULATE SCALE_POS_WEIGHT FOR IMBALANCE
num_negatives = (y_train == 0).sum()
num_positives = (y_train == 1).sum()
scale_weight = num_negatives / num_positives if num_positives > 0 else 1

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


log_reg_model = LogisticRegression(
    class_weight=None,
    max_iter=10000,
    random_state=42
)



# 5. BUILD AND TRAIN THE XGBOOST MODEL
print(f"Training on {len(X_train)} samples...")
print(f"Validating on {len(X_val)} samples...")
print(f"Testing on {len(X_test)} samples...\n")

start = time.perf_counter()

log_reg_model.fit(X_train_scaled, y_train)

end = time.perf_counter()
training_time = end - start
print(f'Training time: {training_time:.10f} seconds')

Training on 7007 samples...
Validating on 1501 samples...
Testing on 1502 samples...

Training time: 0.0553203000 seconds


In [153]:
# 6. EVALUATE ON TEST SET
print("\nGenerating Classification Report on Test Data...")
start = time.perf_counter()

testing_time = end - start
y_pred = log_reg_model.predict(X_test_scaled)
end = time.perf_counter()
testing_time = end - start
print(f'Testing time: {testing_time:.10f} seconds')
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))


Generating Classification Report on Test Data...
Testing time: 0.0012482000 seconds
              precision    recall  f1-score   support

     Class 0       0.90      0.89      0.90       750
     Class 1       0.90      0.90      0.90       752

    accuracy                           0.90      1502
   macro avg       0.90      0.90      0.90      1502
weighted avg       0.90      0.90      0.90      1502



In [154]:


# 1. PREPARE THE 30,000 ROW DATASET
# (Assuming you created 'unmatched_df' using Option 1 from the previous step)

# Safely drop label and metadata columns
cols_to_drop = ['label']
for col in ['split', 'taxonomy', 'ID','idx','subtype']: 
    if col in unmatched_df.columns:
        cols_to_drop.append(col)
        
X_unseen = unmatched_df.drop(columns=cols_to_drop)
y_unseen = unmatched_df['label']

# Convert to matching data types
X_unseen = X_unseen.astype('float32')
y_unseen = y_unseen.astype('int32') 
X_unseen = scaler.fit_transform(X_unseen)

# 2. MAKE PREDICTIONS
print(f"Testing model on {len(X_unseen)} unseen samples...\n")
y_pred_unseen = svm_model.predict(X_unseen)

# 3. EVALUATE
print("--- Confusion Matrix ---")
cm = confusion_matrix(y_unseen, y_pred_unseen)
print(f"True Negatives (Correctly predicted as Normal): {cm[0][0]}")
if cm.shape[1] > 1:
    print(f"False Positives (Mistakenly predicted as Attack): {cm[0][1]}")
else:
    print("False Positives (Mistakenly predicted as Attack): 0")

print("\n--- Classification Report ---")
# We use zero_division=0 to prevent warnings since there are no Class 1 samples in the true labels
print(classification_report(y_unseen, y_pred_unseen, zero_division=0))

Testing model on 30000 unseen samples...

--- Confusion Matrix ---
True Negatives (Correctly predicted as Normal): 15722
False Positives (Mistakenly predicted as Attack): 14278

--- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.52      0.69     30000
           1       0.00      0.00      0.00         0

    accuracy                           0.52     30000
   macro avg       0.50      0.26      0.34     30000
weighted avg       1.00      0.52      0.69     30000

